# Ekstraksi Tabel PDF — per Mata Kuliah

Notebook ini memproses kumpulan PDF digital, mencari tabel yang memuat kolom **Mata Kuliah**, **CLO**, **Indikator Ketercapaian CLO (SUB_CLO)**, dan **Media Pembelajaran**, lalu menggabungkan isinya menjadi satu baris per mata kuliah.

## Cara menggunakan
1. Buka notebook ini di Google Colab.
2. Buat folder `pdf_input` di `MyDrive`, lalu unggah semua PDF ke folder tersebut.
3. Jalankan sel dari atas ke bawah. Hasil akan tersimpan di folder `pdf_output` di Google Drive.

- Ekstraksi utama menggunakan Camelot dengan mode `lattice` (untuk tabel bergaris). Jika tidak menemukan tabel/header yang cocok pada halaman, notebook mencoba `pdfplumber`.
- Nama header dapat disesuaikan pada bagian konfigurasi alias.
- Teks beberapa baris dalam satu sel dipertahankan. Nilai dari beberapa baris untuk satu mata kuliah digabung dengan baris baru.
- File Excel memiliki sheet `hasil` dan `perlu_dicek`; file CSV berisi sheet hasil dalam format tabel.

> **Penting:** hasil dari PDF yang formatnya sangat berbeda tetap perlu diperiksa. Sel gabungan ditangani dengan meneruskan nilai kosong pada baris lanjutan dalam tabel; ubah opsi konfigurasi bila perilaku ini tidak sesuai dengan dokumen Anda.

In [ ]:
# Instalasi dependensi untuk Google Colab
!apt-get -qq update
!apt-get -qq install -y ghostscript
!pip -q install 'camelot-py[cv]' pdfplumber pandas openpyxl

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

# Ubah folder ini sesuai lokasi file di Google Drive.
INPUT_DIR = Path('/content/drive/MyDrive/pdf_input')
OUTPUT_DIR = Path('/content/drive/MyDrive/pdf_output')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Jika True, sel kosong pada baris lanjutan akan diisi dari baris sebelumnya
# di dalam tabel yang sama (berguna untuk sel yang di-merge).
FILL_MERGED_CELLS = True

# Minimal jumlah header target yang harus dikenali agar tabel diproses.
MIN_MATCHED_HEADERS = 3

print('Folder input :', INPUT_DIR)
print('Folder output:', OUTPUT_DIR)
print('PDF ditemukan:', len(list(INPUT_DIR.glob('*.pdf'))) if INPUT_DIR.exists() else 0)

In [ ]:
import re
import unicodedata
from collections import defaultdict

import camelot
import pandas as pd
import pdfplumber

TARGETS = ['mata_kuliah', 'clo', 'sub_clo', 'media_pembelajaran']
DISPLAY_NAMES = {
    'mata_kuliah': 'Mata Kuliah',
    'clo': 'CLO',
    'sub_clo': 'Indikator Ketercapaian CLO (SUB_CLO)',
    'media_pembelajaran': 'Media Pembelajaran',
}

# Tambahkan variasi header yang ditemukan di dokumen Anda bila diperlukan.
HEADER_ALIASES = {
    'mata_kuliah': ['mata kuliah', 'nama mata kuliah', 'mata kuliah (mk)', 'course name', 'course', 'mk'],
    'clo': ['clo', 'course learning outcome', 'capaian pembelajaran mata kuliah'],
    'sub_clo': ['sub clo', 'sub_clo', 'sub-clo', 'indikator ketercapaian clo', 'indikator pencapaian clo', 'indikator ketercapaian', 'sub capaian pembelajaran'],
    'media_pembelajaran': ['media pembelajaran', 'media pembelajaran yang digunakan', 'metode/media pembelajaran', 'learning media'],
}

def clean_cell(value):
    if value is None:
        return ''
    text = str(value).replace('\r', '\n')
    lines = [re.sub(r'\s+', ' ', line).strip() for line in text.split('\n')]
    return '\n'.join(line for line in lines if line)

def normalize_header(value):
    text = unicodedata.normalize('NFKD', clean_cell(value)).encode('ascii', 'ignore').decode('ascii').lower()
    return re.sub(r'[^a-z0-9]+', '', text)

NORMALIZED_ALIASES = {
    key: [normalize_header(alias) for alias in aliases]
    for key, aliases in HEADER_ALIASES.items()
}

def match_header(value):
    norm = normalize_header(value)
    if not norm:
        return None
    # Prioritaskan token SUB_CLO pada header bertingkat seperti 'CLO / SUB_CLO'.
    if 'subclo' in norm:
        return 'sub_clo'
    # Cocokkan persis terlebih dahulu agar 'SUB_CLO' tidak dianggap sebagai 'CLO'.
    for target, aliases in NORMALIZED_ALIASES.items():
        if norm in aliases:
            return target
    # Partial match hanya untuk alias cukup panjang supaya tidak terlalu mudah salah cocok.
    matches = []
    for target, aliases in NORMALIZED_ALIASES.items():
        for alias in aliases:
            if len(alias) >= 8 and alias in norm:
                matches.append((len(alias), target))
    return max(matches)[1] if matches else None

def find_header(rows):
    # Coba gabungkan sampai 3 baris header (misalnya header bertingkat).
    best = None
    max_rows = min(len(rows), 10)
    for start in range(max_rows):
        for height in range(1, min(3, max_rows - start) + 1):
            width = max((len(row) for row in rows[start:start + height]), default=0)
            mapping = {}
            used_columns = set()
            for col in range(width):
                parts = [clean_cell(rows[r][col]) if col < len(rows[r]) else '' for r in range(start, start + height)]
                label = ' '.join(part for part in parts if part)
                target = match_header(label)
                if target and target not in mapping and col not in used_columns:
                    mapping[target] = col
                    used_columns.add(col)
            score = len(mapping)
            candidate = (score, -height, -start, mapping, start + height)
            if best is None or candidate[:3] > best[:3]:
                best = candidate
    if best is None:
        return {}, 0, 0
    return best[3], best[4], best[0]

In [ ]:
def as_rows(table_data):
    rows = []
    for raw_row in table_data or []:
        row = [clean_cell(value) for value in raw_row]
        if any(row):
            rows.append(row)
    return rows

def extract_page_candidates(pdf_path, page_number):
    candidates = []
    # Coba kedua mode Camelot. Stream membantu saat garis tabel tidak terbaca
    # sebagai vektor meskipun tabel tampak bergaris di layar.
    for flavor in ['lattice', 'stream']:
        try:
            camelot_tables = camelot.read_pdf(str(pdf_path), pages=str(page_number), flavor=flavor)
            for table in camelot_tables:
                rows = as_rows(table.df.values.tolist())
                if rows:
                    candidates.append((f'camelot-{flavor}', rows))
        except Exception:
            pass

    # Selalu uji dua strategi pdfplumber, bukan hanya saat Camelot gagal total.
    try:
        with pdfplumber.open(pdf_path) as pdf:
            page = pdf.pages[page_number - 1]
            settings_by_strategy = {
                'lines': {
                    'vertical_strategy': 'lines', 'horizontal_strategy': 'lines',
                    'snap_tolerance': 4, 'join_tolerance': 4,
                },
                'text': {
                    'vertical_strategy': 'text', 'horizontal_strategy': 'text',
                    'min_words_vertical': 2, 'min_words_horizontal': 1,
                    'text_x_tolerance': 3, 'text_y_tolerance': 3,
                },
            }
            for strategy, settings in settings_by_strategy.items():
                try:
                    tables = page.extract_tables(table_settings=settings) or []
                    for table_data in tables:
                        rows = as_rows(table_data)
                        if rows:
                            candidates.append((f'pdfplumber-{strategy}', rows))
                except Exception:
                    pass
    except Exception:
        pass
    return candidates

def preview_table(rows, limit=8):
    lines = []
    for row in rows[:limit]:
        cells = [clean_cell(cell).replace('\n', ' / ')[:100] for cell in row]
        lines.append(' || '.join(cells))
    return ' ### '.join(lines)[:1500]
def rows_from_table(rows, mapping, header_end, pdf_name, page_number):
    output = []
    previous = {target: '' for target in TARGETS}
    for row in rows[header_end:]:
        if not any(clean_cell(value) for value in row):
            continue
        current = {}
        for target in TARGETS:
            col = mapping.get(target)
            current[target] = clean_cell(row[col]) if col is not None and col < len(row) else ''

        course = current['mata_kuliah']
        if not course and not FILL_MERGED_CELLS:
            continue
        if course and course != previous['mata_kuliah']:
            # Jangan membawa CLO/media dari mata kuliah sebelumnya ke mata kuliah baru.
            previous = {target: '' for target in TARGETS}
            previous['mata_kuliah'] = course
        elif not course and FILL_MERGED_CELLS:
            current['mata_kuliah'] = previous['mata_kuliah']

        for target in TARGETS:
            if current[target]:
                previous[target] = current[target]
            elif FILL_MERGED_CELLS and previous[target]:
                current[target] = previous[target]

        if not current['mata_kuliah']:
            continue
        output.append({
            **current,
            'file_sumber': pdf_name,
            'halaman': page_number,
        })
    return output

In [ ]:
def unique_in_order(values):
    found = set()
    result = []
    for value in values:
        value = clean_cell(value)
        if value and value not in found:
            found.add(value)
            result.append(value)
    return '\n'.join(result)

def process_pdf(pdf_path):
    extracted_rows = []
    review_rows = []
    pdf_name = pdf_path.name
    try:
        with pdfplumber.open(pdf_path) as pdf:
            page_count = len(pdf.pages)
    except Exception as exc:
        return [], [{'file_sumber': pdf_name, 'halaman': '', 'catatan': f'PDF tidak dapat dibuka: {exc}', 'pratinjau': ''}]

    for page_number in range(1, page_count + 1):
        candidates = extract_page_candidates(pdf_path, page_number)
        valid_by_method = defaultdict(list)
        candidate_info = []
        for method, rows in candidates:
            mapping, header_end, score = find_header(rows)
            parsed = []
            if score >= MIN_MATCHED_HEADERS and 'mata_kuliah' in mapping:
                parsed = rows_from_table(rows, mapping, header_end, pdf_name, page_number)
                if parsed:
                    valid_by_method[method].append((score, parsed))
            candidate_info.append({
                'method': method, 'score': score, 'mapping': mapping,
                'parsed_count': len(parsed), 'preview': preview_table(rows),
            })

        if valid_by_method:
            # Pilih strategi dengan baris data terbanyak agar tabel header-saja
            # dari satu extractor tidak mengalahkan hasil fallback yang berisi data.
            best_method = max(
                valid_by_method,
                key=lambda method: (
                    sum(len(parsed) for _, parsed in valid_by_method[method]),
                    sum(score for score, _ in valid_by_method[method]),
                ),
            )
            for score, parsed in valid_by_method[best_method]:
                extracted_rows.extend(parsed)
                mapping = next((info['mapping'] for info in candidate_info if info['method'] == best_method and info['score'] == score), {})
                missing = [DISPLAY_NAMES[key] for key in TARGETS if key not in mapping]
                if missing:
                    review_rows.append({
                        'file_sumber': pdf_name, 'halaman': page_number,
                        'catatan': f"Header sebagian terdeteksi via {best_method}; kolom belum ditemukan: {', '.join(missing)}",
                        'pratinjau': '',
                    })
        elif candidate_info:
            best = max(candidate_info, key=lambda info: info['score'])
            if best['score'] >= MIN_MATCHED_HEADERS and 'mata_kuliah' in best['mapping']:
                note = 'Header dikenali, tetapi tidak ada nilai Mata Kuliah pada baris data setelah header.'
            else:
                note = f"Tabel terbaca via {best['method']}, tetapi header target tidak cocok ({best['score']} dari 4 kolom). Periksa alias header."
            review_rows.append({
                'file_sumber': pdf_name, 'halaman': page_number,
                'catatan': note, 'pratinjau': best['preview'],
            })
        else:
            try:
                with pdfplumber.open(pdf_path) as pdf:
                    page_text = clean_cell(pdf.pages[page_number - 1].extract_text() or '')[:1500]
            except Exception:
                page_text = ''
            review_rows.append({
                'file_sumber': pdf_name, 'halaman': page_number,
                'catatan': 'Tidak ada tabel yang terdeteksi. Pratinjau teks halaman membantu membedakan PDF teks dan scan; bila scan, OCR diperlukan.',
                'pratinjau': page_text,
            })

    if not extracted_rows:
        review_rows.append({
            'file_sumber': pdf_name, 'halaman': '',
            'catatan': 'Tidak ada baris data yang berhasil diekstrak dari PDF ini.', 'pratinjau': '',
        })
    return extracted_rows, review_rows

def aggregate_per_course(rows):
    if not rows:
        return pd.DataFrame(columns=['File Sumber', *[DISPLAY_NAMES[key] for key in TARGETS], 'Halaman'])
    df = pd.DataFrame(rows)
    grouped = []
    # Kursus dari PDF berbeda tetap dipisah agar sumbernya tidak tercampur.
    for (file_name, course), group in df.groupby(['file_sumber', 'mata_kuliah'], sort=False, dropna=False):
        item = {'File Sumber': file_name, DISPLAY_NAMES['mata_kuliah']: course}
        for target in ['clo', 'sub_clo', 'media_pembelajaran']:
            item[DISPLAY_NAMES[target]] = unique_in_order(group[target].tolist())
        pages = list(dict.fromkeys(int(p) for p in group['halaman'].tolist() if pd.notna(p)))
        item['Halaman'] = ', '.join(map(str, pages))
        grouped.append(item)
    return pd.DataFrame(grouped)

pdf_files = sorted(INPUT_DIR.glob('*.pdf'))
if not pdf_files:
    raise FileNotFoundError(f'Tidak ada PDF di {INPUT_DIR}. Unggah PDF ke folder tersebut, lalu jalankan ulang.')

all_rows = []
all_reviews = []
for index, pdf_path in enumerate(pdf_files, start=1):
    print(f'[{index}/{len(pdf_files)}] Memproses {pdf_path.name} ...')
    rows, reviews = process_pdf(pdf_path)
    all_rows.extend(rows)
    all_reviews.extend(reviews)

result_df = aggregate_per_course(all_rows)
review_df = pd.DataFrame(all_reviews, columns=['file_sumber', 'halaman', 'catatan', 'pratinjau'])
print(f'\nBaris hasil (per mata kuliah): {len(result_df)}')
print(f'Catatan untuk dicek: {len(review_df)}')
display(result_df.head(10))
if not review_df.empty:
    display(review_df.head(20))
if result_df.empty:
    print('Periksa sheet perlu_dicek di Excel; pratinjau tabel yang terbaca tercantum di sana.')

In [ ]:
excel_path = OUTPUT_DIR / 'hasil_ekstraksi_pdf.xlsx'
csv_path = OUTPUT_DIR / 'hasil_ekstraksi_pdf.csv'

result_df.to_csv(csv_path, index=False, encoding='utf-8-sig')
with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
    result_df.to_excel(writer, index=False, sheet_name='hasil')
    review_df.to_excel(writer, index=False, sheet_name='perlu_dicek')

print('Excel:', excel_path)
print('CSV  :', csv_path)
print('Buka folder pdf_output di Google Drive untuk mengambil hasil.')